In [1]:
import logging
import os
from functools import partial
from pathlib import Path

import hydra
import torch
import wandb
from omegaconf import DictConfig, OmegaConf
from torch.utils.data import DataLoader
import sys

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))

if project_root not in sys.path:
    sys.path.append(project_root)

# Ora puoi importare normalmente usando la notazione con i punti
from src.bev.data.dataset import BEVQADataset, build_tokenizer, collate_train, collate_eval
from src.bev.models.model import BEVVLM, BEVVLMConfig
from src.bev.training.train import train_epoch, val_loss, evaluate

import hydra
from hydra import initialize, compose
from omegaconf import OmegaConf

with initialize(version_base=None, config_path="../configs"):

    cfg = compose(config_name="config", overrides=[
        "training.batch_size=8",  # Esempio: sovrascrivi un parametro al volo
        "training.num_workers=4"
    ])

log = logging.getLogger(__name__)

device = "cuda" if torch.cuda.is_available() else "cpu"

feat_dir = Path(cfg.paths.bev_features_dir)
dict_dir = Path(cfg.paths.dict_dir)
assert feat_dir.exists(), f"Features non trovate -> {feat_dir}"
assert dict_dir.exists(), f"Dict non trovato -> {dict_dir}"

n_bev = cfg.model.num_queries

# Tokenizer (+ token <|bev|>)
tok, bev_token_id = build_tokenizer(cfg.llm.name)

# Dataset
train_ds = BEVQADataset(
    feat_dir / "train", dict_dir / "NuScenes_train_questions.json",
    tok, n_bev_tokens=n_bev, max_len=cfg.training.max_len,
    cache_dir=Path(cfg.paths.dataset_dir) / "cache", fraction=cfg.data.train_fraction,
)
val_ds = BEVQADataset(
    feat_dir / "val", dict_dir / "NuScenes_val_questions.json",
    tok, n_bev_tokens=n_bev, max_len=cfg.training.max_len,
    cache_dir=Path(cfg.paths.dataset_dir) / "cache", fraction=cfg.data.val_fraction,
)
log.info(f"Train: {len(train_ds)} QA | Val: {len(val_ds)} QA | bev_tokens={n_bev}")

pad = tok.pad_token_id
train_loader = DataLoader(
    train_ds, batch_size=cfg.training.batch_size, shuffle=True,
    num_workers=cfg.training.num_workers, pin_memory=True,
    collate_fn=partial(collate_train, pad_id=pad),
)
val_lm_loader = DataLoader(
    val_ds, batch_size=cfg.training.batch_size, shuffle=False,
    num_workers=cfg.training.num_workers,
    collate_fn=partial(collate_train, pad_id=pad),
)

/home/robesafe/BEV/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Tokenizing NuScenes_val_questions: 100%|██████████| 5465/5465 [00:03<00:00, 1518.72it/s]


In [3]:
# Mappa i loader per fare un ciclo pulito stampando i loro nomi
loaders = {
    "Train Loader": train_loader,
    "Val LM Loader": val_lm_loader,
}

print("=== BATCH EXAMPLE (Right-Padding) ===\n")

for name, loader in loaders.items():
    print(f"{'='*10} {name.upper()} {'='*10}")
    
    try:
        batch = next(iter(loader))
    except StopIteration:
        print(f"Il loader {name} è vuoto!\n")
        continue
        
    # Estrae i tensori principali
    input_ids = batch.get("input_ids")
    labels = batch.get("labels")
    attention_mask = batch.get("attention_mask")
    
    if input_ids is None:
        print(f"Chiave 'input_ids' non trovata in {name}.\n")
        continue
        
    n_elements = min(5, input_ids.shape[0])
    print(f"Batch shape: {list(input_ids.shape)} (Total length in the batch sequence: {input_ids.shape[1]})")
    print(f"Sequence of the first {n_elements} items of the batch:\n")
    
    for i in range(n_elements):
        print(f"--- Item {i+1} ---")
        
        ids_list = input_ids[i].tolist()
        mask_list = attention_mask[i].tolist() if attention_mask is not None else None
        
        # Stampiamo le liste COMPLETE (senza tagli) per vedere il padding a destra
        print(f"input_ids:      {ids_list}")
        
        if labels is not None:
            labels_list = labels[i].tolist()
            print(f"labels:         {labels_list}")
        else:
            print("labels:         [Non presenti in questo Loader]")
            
        if mask_list is not None:
            print(f"attention_mask: {mask_list}")
        
        # Decodifica l'INTERA lista inclusi i pad token speciali per vederli a schermo
        try:
            decoded_text = tok.decode(ids_list, skip_special_tokens=False)
            print(f"Text:\n\"{decoded_text}\"")
        except Exception as e:
            print(f"[Impossibile decodificare il testo: {e}]")
            
        print()
    print("\n")

=== BATCH EXAMPLE (Right-Padding) ===

========== TRAIN LOADER ==========
Batch shape: [8, 181] (Total length in the batch sequence: 181)
Sequence of the first 5 items of the batch:

--- Item 1 ---
input_ids:      [151644, 8948, 198, 2610, 525, 264, 9842, 17847, 13, 1446, 525, 2661, 264, 11958, 594, 46697, 22503, 4565, 2415, 315, 279, 6109, 2163, 279, 36274, 12, 19764, 11, 11170, 279, 14590, 11474, 13, 21806, 279, 3405, 911, 279, 6109, 304, 264, 2805, 11652, 13, 151645, 198, 151644, 872, 198, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665, 151665